# 🤖 Evaluasi Kinerja Auto-Scaler
Meniru evaluasi elastisitas Auto-Scaler. Kami memuat hasil prediksi yang dihasilkan masing-masing model untuk disimulasikan.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import os

# Load Real Test Data & Scaler
data = np.load('processed_data.npz')
y_test = data['y_test']

with open('cpu_scaler.pkl', 'rb') as f:
    cpu_scaler = pickle.load(f)

# Load Predictions if available
model_names = ['LSTM', 'GRU', 'BiLSTM', 'DUCFF']
predictions = {}
for name in model_names:
    if os.path.exists(f'pred_{name}.npy'):
        predictions[name] = np.load(f'pred_{name}.npy')
        print(f"Loaded predictions for {name}")
    else:
        print(f"File pred_{name}.npy not found. Skip.")


In [ ]:
y_test_real = cpu_scaler.inverse_transform(y_test.reshape(-1, 1)).flatten()
CPU_CAPACITY_PER_CONTAINER = 2.0
R_min = 5
CDT_MAX = 10  
SDR = 0.40  

def simulate_autoscaler(predicted_cpu, actual_cpu):
    R_current = R_min; CDT = 0; supply_history = []; demand_history = []
    for t in range(len(predicted_cpu)):
        R_demand = max(int(np.ceil(actual_cpu[t] / CPU_CAPACITY_PER_CONTAINER)), R_min)
        demand_history.append(R_demand)
        R_estimated = max(int(np.ceil(predicted_cpu[t] / CPU_CAPACITY_PER_CONTAINER)), R_min)
        
        if R_estimated > R_current:
            R_current = R_estimated
            CDT = CDT_MAX
        elif R_estimated < R_current:
            if CDT <= 0:
                CDT = CDT_MAX
                R_current = max(R_current - int(np.floor((R_current - R_estimated) * (1 - SDR))), R_estimated, R_min)
            else:
                CDT -= 1
        else:
            if CDT > 0: CDT -= 1
        supply_history.append(R_current)
    return np.array(supply_history), np.array(demand_history)

total_T = len(y_test_real)
demand_real = np.array([max(int(np.ceil(d / CPU_CAPACITY_PER_CONTAINER)), R_min) for d in y_test_real])
reactive_supply = np.zeros(total_T)
reactive_supply[0] = R_min
for t in range(1, total_T):
    reactive_supply[t] = max(int(np.ceil(y_test_real[t-1] / CPU_CAPACITY_PER_CONTAINER)), R_min)

def calc_metrics(supply, demand):
    under = np.maximum(demand - supply, 0)
    over = np.maximum(supply - demand, 0)
    safe_demand = np.where(demand == 0, 1, demand)
    return ((100.0/total_T)*np.sum(under/safe_demand), (100.0/total_T)*np.sum(over/safe_demand), (np.sum(under>0)/total_T)*100.0, (np.sum(over>0)/total_T)*100.0)

theta_U_n, theta_O_n, T_U_n, T_O_n = calc_metrics(reactive_supply, demand_real)
autoscaler_metrics = {'Reactive': {'θ_U': theta_U_n, 'θ_O': theta_O_n, 'T_U': T_U_n, 'T_O': T_O_n, 'η (Speedup)': 1.00}}

all_supplies = {}
if len(predictions) > 0:
    for name, pred in predictions.items():
        pred_real = cpu_scaler.inverse_transform(pred.reshape(-1, 1)).flatten()
        supply, demand = simulate_autoscaler(pred_real, y_test_real)
        all_supplies[name] = supply
        t_U_a, t_O_a, TU_a, TO_a = calc_metrics(supply, demand)
        eta = ((theta_U_n/t_U_a if t_U_a>0 else 1)*(theta_O_n/t_O_a if t_O_a>0 else 1)*(T_U_n/TU_a if TU_a>0 else 1)*(T_O_n/TO_a if TO_a>0 else 1))**0.25
        autoscaler_metrics[name] = {'θ_U': t_U_a, 'θ_O': t_O_a, 'T_U': TU_a, 'T_O': TO_a, 'η (Speedup)': eta}

    display(pd.DataFrame(autoscaler_metrics).T)
else:
    print("Prediksi belum ada, silahkan jalankan notebook model terlebih dahulu.")


## Total Started Containers (Overhead / API Oscillations)

In [ ]:
if len(all_supplies) > 0:
    started_containers = {}
    for name, supply in all_supplies.items():
        started_containers[name] = sum((supply[i] - supply[i-1]) for i in range(1, len(supply)) if supply[i] > supply[i-1])

    plt.figure(figsize=(8, 4))
    sns.barplot(x=list(started_containers.keys()), y=list(started_containers.values()), palette='coolwarm')
    plt.title('Total Containers Started (Overhead)', fontsize=14)
    plt.show()


## Qualitative Analysis of Demand vs Supply (GDS Algorithm)

In [ ]:
if len(all_supplies) > 0:
    fig, axes = plt.subplots(len(all_supplies), 1, figsize=(14, 5 * len(all_supplies)), sharex=True)
    plot_len = 400
    
    # Kalo cuma ada 1 model, axes mungkin bukan list, jadi kita akali:
    if len(all_supplies) == 1:
        axes = [axes]

    for i, (m_name, supply) in enumerate(all_supplies.items()):
        axes[i].plot(demand_real[:plot_len], label='Demand (Red)', color='red', linestyle='--')
        axes[i].plot(supply[:plot_len], label=f'Supply {m_name} (Blue)', color='blue', drawstyle='steps-post', alpha=0.7)
        axes[i].set_title(f'({m_name}): Auto-Scaler Supply vs Demand', fontsize=12)
        axes[i].legend()

    plt.tight_layout()
    plt.show()
